# 制造业 AI 演示 笔记本
## 不依赖 MES，如何从原始设备日志中提取“良率相关”洞察

### Notebook 定位（本次交付重点）
本 Notebook 的目标不是直接对原始设备 Log 做分析，而是先将设备导出的原始宽表 Log 转换为适合良率分析、异常检测和 AI 洞察的标准化分析数据表。

原始设备 Log 中，L1 和 L2（以及可能更多线体）相关字段以列名前缀形式横向展开。本 Notebook 会将线体前缀信息从列名中剥离并转换为行级 `Line` 字段，统一为同一套标准字段口径，避免把 L1 和 L2 当成两套割裂数据。

标准化输出文件（后续分析统一入口）：`data/processed_equipment_log_standardized.xlsx`。

### 项目目标与数据链路
```text
原始设备 Log Excel
    ↓
字段读取与重复列处理
    ↓
L1 / L2 字段拆分
    ↓
统一字段命名
    ↓
新增 Line 字段
    ↓
生成标准化数据文件
    ↓
特征选择与清洗
    ↓
EDA / 异常检测 / 良率影响分析 / AI 洞察
```

> 说明：原始设备 Log 更适合设备记录，不适合直接建模。标准化后的数据才是后续分析可信输入；后续分析层应基于标准化结果，而不是回退原始宽表。

### 业务背景（前因）
在很多工厂里，设备每天都会产生大量日志（上百列参数、秒级/分钟级时间序列），但现场团队经常遇到同一个问题：
 - 数据很多，但难以快速回答“最近良率波动可能和哪些过程参数相关？”



In [ ]:
import re
from pathlib import Path
import openpyxl

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.style.use("seaborn-v0_8-whitegrid")
pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 140)

# 中文显示名映射：仅用于图表和展示，不修改原始字段名
column_display_map = {
    "running speed": "运行速度",
    "speed": "速度",
    "line speed": "产线速度",
    "ag deplating ph": "脱银槽pH值",
    "ph": "pH值",
    "pressure": "压力",
    "temperature": "温度",
    "temp": "温度",
    "flow": "流量",
    "current": "电流",
    "voltage": "电压",
    "torque": "扭矩",
    "vibration": "振动",
    "rpm": "转速",
    "speed_diff": "速度变化强度",
    "ph_range": "pH波动范围",
    "pressure_variation": "压力波动",
    "anomaly_score": "异常分数",
    "anomaly_flag": "异常标记",
    "normal_mean": "正常段均值",
    "abnormal_mean": "异常段均值",
    "abs_diff": "绝对差异",
    "pct_diff_vs_normal": "相对正常段变化(%)",
}

def cn_label(col_name):
    """返回中文展示名；找不到时回退原列名。"""
    return column_display_map.get(str(col_name).strip().lower(), col_name)


In [ ]:
# 中文字体初始化（用于 Matplotlib 图表）
import matplotlib as mpl
from matplotlib import font_manager
import warnings

# 可选：隐藏缺字形告警，避免演示时出现红色 warning
warnings.filterwarnings("ignore", category=UserWarning, message=r"Glyph.*missing from font")

# 按优先级尝试字体：PingFang（macOS）-> STHeiti -> Hiragino Sans GB -> Arial Unicode
font_candidates = [
    ("PingFang SC", [
        "/System/Library/Fonts/PingFang.ttc",
    ]),
    ("STHeiti", [
        "/System/Library/Fonts/STHeiti Light.ttc",
        "/System/Library/Fonts/STHeiti Medium.ttc",
    ]),
    ("Hiragino Sans GB", [
        "/System/Library/Fonts/Hiragino Sans GB.ttc",
    ]),
    ("Arial Unicode MS", [
        "/Library/Fonts/Arial Unicode.ttf",
        "/System/Library/Fonts/Supplemental/Arial Unicode.ttf",
        "/System/Library/Fonts/Supplemental/Arial Unicode MS.ttf",
    ]),
]

selected_font = None
for font_name, font_paths in font_candidates:
    for font_path in font_paths:
        p = Path(font_path)
        if p.exists():
            try:
                font_manager.fontManager.addfont(str(p))
                selected_font = font_name
                break
            except Exception as e:
                print(f"加载字体失败 {font_path}: {e}")
    if selected_font:
        break

if selected_font:
    mpl.rcParams["font.family"] = selected_font
    mpl.rcParams["font.sans-serif"] = [selected_font]
    mpl.rcParams["axes.unicode_minus"] = False
    print(f"Matplotlib 中文字体已设置为: {selected_font}")
else:
    print("未找到可用中文字体（PingFang/STHeiti/Hiragino Sans GB/Arial Unicode）。图表可能出现中文缺字。")



## 1) 数据加载
此步骤的关键点：
- 兼容 Excel / CSV；
- 尽量自动识别时间列并排序；
- 先建立“可复用的数据读取函数”，后续项目可直接复用。

**为什么这样做？**
现场数据常来自不同设备厂商，字段命名不统一。先把加载层做稳，后续分析才不会反复返工。


In [ ]:
def load_data(file_path, time_col_candidates=None):
    # 中文说明：统一入口读取设备日志，支持 Excel/CSV。
    # 前因：客户现场常见多种格式导出。
    # 后果：统一函数后，后续清洗/建模代码不需要改动。
    file_path = Path(file_path)
    if not file_path.exists():
        raise FileNotFoundError(f"文件不存在：{file_path}")

    if time_col_candidates is None:
        # 中文说明：常见时间列候选名（可按客户现场再扩展）
        time_col_candidates = ["timestamp", "time", "datetime", "date_time", "record_time", "log_time"]

    if file_path.suffix.lower() in [".xlsx", ".xls"]:
        df = pd.read_excel(file_path)
    elif file_path.suffix.lower() == ".csv":
        df = pd.read_csv(file_path)
    else:
        raise ValueError("不支持的文件类型，请使用 Excel（.xlsx/.xls）或 CSV。")

    # 中文说明：尝试自动匹配时间列，并做时间解析+排序
    normalized_map = {c.lower().strip(): c for c in df.columns}
    selected_time_col = None
    for cand in time_col_candidates:
        if cand in normalized_map:
            selected_time_col = normalized_map[cand]
            break

    if selected_time_col is not None:
        df[selected_time_col] = pd.to_datetime(df[selected_time_col], errors="coerce")
        df = df.sort_values(selected_time_col).reset_index(drop=True)
        print(f"已识别时间列：{selected_time_col}")
    else:
        print("未在候选字段中识别到时间列。")

    print(f"数据规模：{df.shape[0]:,} 行 × {df.shape[1]:,} 列")
    return df

In [ ]:
# --- 中文说明：请改成客户现场日志文件路径 ---
# 示例：data_path = "data/equipment_log.xlsx"
# 为方便快速演示，这里默认读取仓库内的样例文件。
from pathlib import Path

data_path = Path("../data/equipment_log.xlsx").resolve()
print("当前读取文件：", data_path)
df_raw = load_data(data_path)

# 展示原始数据前几行与字段统计，帮助工程师快速建立数据直觉
display(df_raw.head(3))
display(df_raw.describe(include="all").transpose().head(10))


## 1.5) 设备日志标准化处理（新增）
在数据加载后、异常检测前新增标准化模块，自动识别 Time / 线体字段 / 公共字段，并输出标准化结果。


## 1.6) 原始数据结构问题说明（为什么必须先标准化）
原始数据在工程记录上是完整的，但直接用于良率分析通常有以下障碍：

1. L1 / L2 字段横向展开，同一含义字段被拆成两套列；
2. 字段名带有线体前缀，不利于统一统计口径；
3. 可能存在重复字段或近似重复字段（读取后会出现后缀）；
4. 个别字段只存在于某一线体（如仅 L2 存在）；
5. 原始格式适合设备记录，不适合直接做建模与跨线对比。

因此本 Notebook 先完成结构标准化，再进入特征选择与后续分析。


In [ ]:
# ============================================================
# 设备日志标准化处理 Cell
# 功能：
# - 宽表设备日志 -> Line_Snapshot / Common_Snapshot / Measurement_Long
# - 自动识别 L1/L2/L3...
# - 自动识别公共设备参数
# - 自动保留重复字段
# - 自动输出标准化 Excel（data 目录）
# - 输入文件未变化时跳过重复生成
# ============================================================

import re
import json
import itertools
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd

from IPython.display import display, Markdown


# -----------------------------
# 0. 输入输出文件设置
# -----------------------------
INPUT_FILE = Path(str(data_path)) if "data_path" in globals() else Path("Y2026M04D20data V1.0.xlsx")
OUTPUT_DIR = Path("data")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_FILE = OUTPUT_DIR / "processed_equipment_log_standardized.xlsx"
CACHE_META_FILE = OUTPUT_DIR / ".processed_equipment_log_standardized.meta.json"


def build_input_signature(file_path):
    file_path = Path(file_path)
    stat = file_path.stat()
    return {
        "path": str(file_path.resolve()),
        "size": int(stat.st_size),
        "mtime_ns": int(stat.st_mtime_ns),
    }


def should_rebuild(input_file, output_file, cache_meta_file):
    input_file = Path(input_file)
    output_file = Path(output_file)
    cache_meta_file = Path(cache_meta_file)

    if not input_file.exists():
        raise FileNotFoundError(f"找不到输入文件：{input_file.resolve()}")

    if not output_file.exists() or not cache_meta_file.exists():
        return True, build_input_signature(input_file)

    try:
        old_meta = json.loads(cache_meta_file.read_text(encoding="utf-8"))
    except Exception:
        return True, build_input_signature(input_file)

    new_signature = build_input_signature(input_file)
    old_signature = old_meta.get("input_signature", {})
    return old_signature != new_signature, new_signature


def save_cache_meta(cache_meta_file, input_signature):
    cache_meta_file = Path(cache_meta_file)
    payload = {
        "input_signature": input_signature,
        "output_file": str(Path(OUTPUT_FILE).resolve()),
    }
    cache_meta_file.write_text(
        json.dumps(payload, ensure_ascii=False, indent=2),
        encoding="utf-8",
    )


def load_standardized_excel(output_file):
    output_file = Path(output_file)
    sheets = pd.read_excel(output_file, sheet_name=None)

    required = [
        "Raw_Data", "Field_Map", "Field_Diagnostics",
        "Duplicate_Value_Check", "Line_Snapshot", "Common_Snapshot", "Measurement_Long",
    ]
    missing = [s for s in required if s not in sheets]
    if missing:
        raise ValueError(f"输出文件缺少必要 Sheet: {missing}")

    return {
        "raw_data": sheets["Raw_Data"],
        "field_map": sheets["Field_Map"],
        "field_diagnostics": sheets["Field_Diagnostics"],
        "duplicate_value_check": sheets["Duplicate_Value_Check"],
        "line_snapshot": sheets["Line_Snapshot"],
        "common_snapshot": sheets["Common_Snapshot"],
        "measurement_long": sheets["Measurement_Long"],
    }


def read_excel_keep_duplicate_headers(file_path):
    file_path = Path(file_path)
    if not file_path.exists():
        raise FileNotFoundError(f"找不到输入文件：{file_path.resolve()}")

    df_raw_all = pd.read_excel(file_path, header=None)
    if df_raw_all.empty:
        raise ValueError("输入 Excel 为空，无法解析。")

    header = df_raw_all.iloc[0].fillna("").astype(str).str.strip().tolist()
    data = df_raw_all.iloc[1:].reset_index(drop=True).copy()

    counts = defaultdict(int)
    unique_cols = []
    meta_rows = []
    for idx, col in enumerate(header, start=1):
        original_name = col if col else f"Unnamed_{idx}"
        counts[original_name] += 1
        duplicate_index = counts[original_name]
        unique_name = original_name if duplicate_index == 1 else f"{original_name}__dup{duplicate_index}"
        unique_cols.append(unique_name)
        meta_rows.append({
            "column_index": idx,
            "original_name": original_name,
            "unique_name": unique_name,
            "duplicate_index": duplicate_index,
        })

    data.columns = unique_cols
    return data, pd.DataFrame(meta_rows)


def parse_field(original_name):
    name = str(original_name).strip()
    if name.lower() == "time":
        return {"scope": "time", "line_id": "", "parameter": "Time"}

    match = re.match(r"^(L\d+)[\s_]+(.+)$", name, flags=re.IGNORECASE)
    if match:
        return {
            "scope": "line",
            "line_id": match.group(1).upper(),
            "parameter": match.group(2).strip(),
        }
    return {"scope": "common", "line_id": "", "parameter": name}


def classify_parameter(parameter):
    p = str(parameter).lower()
    if any(k in p for k in ["mode", "product", "lot", "tool", "electrode"]):
        return "production_context"
    if "speed" in p:
        return "speed"
    if "current" in p:
        return "current"
    if "voltage" in p:
        return "voltage"
    if any(k in p for k in ["tc", "temp", "oven"]):
        return "temperature"
    if "flow" in p:
        return "flow"
    if "pressure" in p:
        return "pressure"
    if "cond" in p:
        return "conductivity"
    if "ph" in p:
        return "ph"
    if "level" in p:
        return "level"
    if any(k in p for k in ["onoff", "on/off", "switch"]):
        return "switch"
    return "other"


def build_field_map(header_meta):
    parsed_df = pd.DataFrame([parse_field(r["original_name"]) for _, r in header_meta.iterrows()])
    field_map = pd.concat([header_meta.reset_index(drop=True), parsed_df], axis=1)
    field_map["Category"] = field_map["parameter"].apply(classify_parameter)
    return field_map


def build_field_diagnostics(field_map):
    diagnostics = []

    duplicate_names = field_map.groupby("original_name").size().reset_index(name="count")
    duplicate_names = duplicate_names[duplicate_names["count"] > 1]
    for _, row in duplicate_names.iterrows():
        diagnostics.append({
            "issue_type": "duplicate_column",
            "severity": "warning",
            "field": row["original_name"],
            "scope": "",
            "line_id": "",
            "detail": f"字段重复出现 {int(row['count'])} 次",
            "suggestion": "暂不删除，自动加后缀保留；后续比较两列数据是否完全一致。",
        })

    line_fields = field_map[field_map["scope"] == "line"].copy()
    line_ids = sorted(line_fields["line_id"].dropna().unique().tolist())
    if len(line_ids) >= 2:
        parameter_by_line = {
            line: set(line_fields[line_fields["line_id"] == line]["parameter"].tolist())
            for line in line_ids
        }
        all_parameters = set().union(*parameter_by_line.values()) if parameter_by_line else set()
        for parameter in sorted(all_parameters):
            existing_lines = [line for line in line_ids if parameter in parameter_by_line[line]]
            missing_lines = [line for line in line_ids if parameter not in parameter_by_line[line]]
            if missing_lines:
                diagnostics.append({
                    "issue_type": "line_parameter_mismatch",
                    "severity": "info",
                    "field": parameter,
                    "scope": "line",
                    "line_id": ",".join(existing_lines),
                    "detail": f"{parameter} 存在于 {existing_lines}，但缺失于 {missing_lines}",
                    "suggestion": "保留已有字段，不强行补齐；建议工程师确认是否为漏导出、命名不一致或设备配置差异。",
                })

    common_fields = field_map[field_map["scope"] == "common"].copy()
    for _, row in common_fields.iterrows():
        diagnostics.append({
            "issue_type": "common_field",
            "severity": "info",
            "field": row["original_name"],
            "scope": "common",
            "line_id": "",
            "detail": "字段无 L1/L2/L3 前缀，已识别为设备公共参数",
            "suggestion": "归入 Common_Snapshot，用于分析槽液、温度、pH、压力、电导率等公共设备状态。",
        })

    return pd.DataFrame(diagnostics)


def build_line_snapshot(df_data, field_map):
    time_rows = field_map[field_map["scope"] == "time"]
    if time_rows.empty:
        raise ValueError("未找到 Time 列，请确认 Excel 表头是否包含 Time。")
    time_col = time_rows.iloc[0]["unique_name"]

    line_map = field_map[field_map["scope"] == "line"].copy()
    line_ids = sorted(line_map["line_id"].dropna().unique().tolist())
    if not line_ids:
        return pd.DataFrame({"Time": df_data[time_col]})

    frames = []
    for line_id in line_ids:
        sub_map = line_map[line_map["line_id"] == line_id].copy()
        temp = pd.DataFrame({"Time": df_data[time_col], "Line": line_id})
        parameter_counter = defaultdict(int)
        for _, row in sub_map.iterrows():
            parameter = row["parameter"]
            parameter_counter[parameter] += 1
            clean_parameter = parameter if parameter_counter[parameter] == 1 else f"{parameter}__dup{parameter_counter[parameter]}"
            temp[clean_parameter] = df_data[row["unique_name"]]
        frames.append(temp)
    return pd.concat(frames, axis=0, ignore_index=True, sort=False)


def build_common_snapshot(df_data, field_map):
    time_rows = field_map[field_map["scope"] == "time"]
    if time_rows.empty:
        raise ValueError("未找到 Time 列，请确认 Excel 表头是否包含 Time。")
    time_col = time_rows.iloc[0]["unique_name"]

    common_map = field_map[field_map["scope"] == "common"].copy()
    out = pd.DataFrame({"Time": df_data[time_col]})

    parameter_counter = defaultdict(int)
    for _, row in common_map.iterrows():
        parameter = row["parameter"]
        parameter_counter[parameter] += 1
        clean_parameter = parameter if parameter_counter[parameter] == 1 else f"{parameter}__dup{parameter_counter[parameter]}"
        out[clean_parameter] = df_data[row["unique_name"]]
    return out


def build_measurement_long(df_data, field_map):
    time_rows = field_map[field_map["scope"] == "time"]
    if time_rows.empty:
        raise ValueError("未找到 Time 列，请确认 Excel 表头是否包含 Time。")
    time_col = time_rows.iloc[0]["unique_name"]

    target_map = field_map[field_map["scope"].isin(["line", "common"])].copy()
    long_parts = []
    line_param_counter = defaultdict(int)
    common_param_counter = defaultdict(int)

    for _, row in target_map.iterrows():
        scope = row["scope"]
        line_id = row["line_id"] if scope == "line" else "machine"
        parameter = row["parameter"]

        if scope == "line":
            key = (line_id, parameter)
            line_param_counter[key] += 1
            dup_idx = line_param_counter[key]
        else:
            key = parameter
            common_param_counter[key] += 1
            dup_idx = common_param_counter[key]

        clean_parameter = parameter if dup_idx == 1 else f"{parameter}__dup{dup_idx}"
        long_parts.append(pd.DataFrame({
            "Time": df_data[time_col],
            "Scope": scope,
            "Line": line_id,
            "Parameter": clean_parameter,
            "Original_Column": row["original_name"],
            "Unique_Column": row["unique_name"],
            "Category": classify_parameter(clean_parameter),
            "Value": df_data[row["unique_name"]],
        }))

    if not long_parts:
        return pd.DataFrame(columns=[
            "Time", "Scope", "Line", "Parameter",
            "Original_Column", "Unique_Column", "Category", "Value",
        ])
    return pd.concat(long_parts, axis=0, ignore_index=True)


def build_duplicate_value_check(df_data, field_map):
    checks = []
    dup_groups = field_map.groupby("original_name")

    for original_name, group in dup_groups:
        if len(group) <= 1:
            continue
        unique_cols = group["unique_name"].tolist()

        for col_a, col_b in itertools.combinations(unique_cols, 2):
            s1 = df_data[col_a]
            s2 = df_data[col_b]
            identical = s1.fillna("__NA__").equals(s2.fillna("__NA__"))

            n1 = pd.to_numeric(s1, errors="coerce")
            n2 = pd.to_numeric(s2, errors="coerce")
            valid = n1.notna() & n2.notna()
            corr = n1[valid].corr(n2[valid]) if valid.sum() >= 2 else np.nan

            if identical:
                assessment = "identical_values"
            elif pd.notna(corr) and corr >= 0.99:
                assessment = "highly_similar"
            elif valid.sum() >= 2:
                assessment = "different_values"
            else:
                assessment = "non_numeric_or_insufficient_data"

            checks.append({
                "original_name": original_name,
                "column_a": col_a,
                "column_b": col_b,
                "identical": bool(identical),
                "corr": corr,
                "non_null_count_a": int(s1.notna().sum()),
                "non_null_count_b": int(s2.notna().sum()),
                "unique_count_a": int(s1.nunique(dropna=True)),
                "unique_count_b": int(s2.nunique(dropna=True)),
                "mean_a": n1.mean(),
                "std_a": n1.std(),
                "mean_b": n2.mean(),
                "std_b": n2.std(),
                "duplicate_value_assessment": assessment,
            })

    return pd.DataFrame(checks)


def export_standardized_excel(output_file, raw_data, field_map, field_diagnostics, duplicate_value_check, line_snapshot, common_snapshot, measurement_long):
    with pd.ExcelWriter(output_file, engine="openpyxl") as writer:
        raw_data.to_excel(writer, sheet_name="Raw_Data", index=False)
        field_map.to_excel(writer, sheet_name="Field_Map", index=False)
        field_diagnostics.to_excel(writer, sheet_name="Field_Diagnostics", index=False)
        duplicate_value_check.to_excel(writer, sheet_name="Duplicate_Value_Check", index=False)
        line_snapshot.to_excel(writer, sheet_name="Line_Snapshot", index=False)
        common_snapshot.to_excel(writer, sheet_name="Common_Snapshot", index=False)
        measurement_long.to_excel(writer, sheet_name="Measurement_Long", index=False)


display(Markdown("## 设备日志标准化处理开始"))
print(f"输入文件：{INPUT_FILE}")
print(f"输出文件：{OUTPUT_FILE}")

need_rebuild, current_input_signature = should_rebuild(INPUT_FILE, OUTPUT_FILE, CACHE_META_FILE)

if need_rebuild:
    print("检测到输入变更（或首次运行），开始重新标准化处理...")
    raw_data, header_meta = read_excel_keep_duplicate_headers(INPUT_FILE)
    field_map = build_field_map(header_meta)
    field_diagnostics = build_field_diagnostics(field_map)
    line_snapshot = build_line_snapshot(raw_data, field_map)
    common_snapshot = build_common_snapshot(raw_data, field_map)
    measurement_long = build_measurement_long(raw_data, field_map)
    duplicate_value_check = build_duplicate_value_check(raw_data, field_map)

    export_standardized_excel(
        OUTPUT_FILE,
        raw_data,
        field_map,
        field_diagnostics,
        duplicate_value_check,
        line_snapshot,
        common_snapshot,
        measurement_long,
    )
    save_cache_meta(CACHE_META_FILE, current_input_signature)
    print(f"已生成文件：{OUTPUT_FILE.resolve()}")
else:
    print("输入文件未变化，跳过重复生成，直接复用已有标准化结果。")
    cached = load_standardized_excel(OUTPUT_FILE)
    raw_data = cached["raw_data"]
    field_map = cached["field_map"]
    field_diagnostics = cached["field_diagnostics"]
    duplicate_value_check = cached["duplicate_value_check"]
    line_snapshot = cached["line_snapshot"]
    common_snapshot = cached["common_snapshot"]
    measurement_long = cached["measurement_long"]
    print(f"复用文件：{OUTPUT_FILE.resolve()}")

display(Markdown("## 处理完成"))
print("====== 数据规模 ======")
print(f"Raw_Data:              {raw_data.shape[0]} 行 × {raw_data.shape[1]} 列")
print(f"Field_Map:             {field_map.shape[0]} 行 × {field_map.shape[1]} 列")
print(f"Field_Diagnostics:     {field_diagnostics.shape[0]} 行 × {field_diagnostics.shape[1]} 列")
print(f"Duplicate_Value_Check: {duplicate_value_check.shape[0]} 行 × {duplicate_value_check.shape[1]} 列")
print(f"Line_Snapshot:         {line_snapshot.shape[0]} 行 × {line_snapshot.shape[1]} 列")
print(f"Common_Snapshot:       {common_snapshot.shape[0]} 行 × {common_snapshot.shape[1]} 列")
print(f"Measurement_Long:      {measurement_long.shape[0]} 行 × {measurement_long.shape[1]} 列")

print("====== 字段作用域统计 ======")
display(field_map["scope"].value_counts().rename_axis("scope").reset_index(name="count"))

print("====== 自动识别到的线体 ======")
detected_lines = sorted(field_map.loc[field_map["scope"] == "line", "line_id"].dropna().unique().tolist())
print(detected_lines)

print("====== 字段分类统计 ======")
display(field_map["Category"].value_counts().rename_axis("Category").reset_index(name="count"))

print("====== 字段诊断类型统计 ======")
if not field_diagnostics.empty:
    display(field_diagnostics["issue_type"].value_counts().rename_axis("issue_type").reset_index(name="count"))
else:
    print("无字段诊断问题。")

print("====== 重点字段诊断：Strike OnOff / Oven(1) TC ======")
if not field_diagnostics.empty:
    key_diag = field_diagnostics[
        field_diagnostics["field"].astype(str).str.contains(r"Strike OnOff|Oven\(1\) TC", case=False, regex=True, na=False)
    ]
    display(key_diag)
else:
    print("无诊断信息。")

print("====== 重复字段数值检查 ======")
if not duplicate_value_check.empty:
    display(duplicate_value_check)
else:
    print("未发现重复字段。")

print("====== Field_Map 前 20 行 ======")
display(field_map.head(20))

print("====== Line_Snapshot 前 10 行 ======")
display(line_snapshot.head(10))

print("====== Common_Snapshot 前 10 行 ======")
display(common_snapshot.head(10))

print("====== Measurement_Long 前 10 行 ======")
display(measurement_long.head(10))

display(Markdown("""
### 标准化结果说明

本 Cell 已将原始设备宽表日志标准化为：

1. **Line_Snapshot**：每个时间点、每条线一行，适合做 L1/L2/L3 对比。
2. **Common_Snapshot**：设备公共参数快照，适合分析槽液、温度、pH、压力、电导率等共享系统状态。
3. **Measurement_Long**：指标长表，适合后续异常检测、趋势图、参数波动排名和 AI 洞察。
4. **Field_Map**：字段映射表，记录每一列属于时间、线体还是公共参数。
5. **Field_Diagnostics**：字段诊断表，识别重复字段、线体字段不一致和公共参数。
6. **Duplicate_Value_Check**：重复字段数值检查，判断重复字段是否完全一致或高度相似。

该结构不依赖固定列号，后续一机三线、一机四线也可以继续复用。
"""))


# 输出摘要：标准化结果是后续特征层统一入口
print(f"标准化数据已输出：{OUTPUT_FILE}")
print(f"标准化主表维度（Line_Snapshot）：{line_snapshot_df.shape[0]} 行 × {line_snapshot_df.shape[1]} 列")


### 字段映射与转换示例（示意）
| 原始字段 | 标准化后字段 | 说明 |
|---|---|---|
| L1 Lot ID | Lot ID | L1 产品批次号 |
| L2 Lot ID | Lot ID | L2 产品批次号 |
| L1 DI(1) Pressure | DI(1) Pressure | L1 工艺参数 |
| L2 DI(1) Pressure | DI(1) Pressure | L2 工艺参数 |

> 实际映射请以标准化输出中的 `Field_Map` Sheet 为准；该表由代码基于当前输入字段动态生成。


## 2) 数据探索分析：基于标准化设备日志
本节不再直接分析原始 173 列宽表，而是基于标准化后的 Line_Snapshot、Common_Snapshot 和 Measurement_Long 进行探索。这样可以兼容一机两线、一机三线或更多线体，并能区分线体参数与设备公共参数。


In [ ]:
# ===== 数据探索分析（基于标准化文件） =====
from pathlib import Path
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

DATA_DIR = Path("data")
STANDARDIZED_EXCEL_FILE = DATA_DIR / "processed_equipment_log_standardized.xlsx"

if not STANDARDIZED_EXCEL_FILE.exists():
    raise FileNotFoundError(
        "未找到 data/processed_equipment_log_standardized.xlsx。请先运行设备日志标准化与缓存模块。"
    )

field_map = pd.read_excel(STANDARDIZED_EXCEL_FILE, sheet_name="Field_Map")
field_diagnostics = pd.read_excel(STANDARDIZED_EXCEL_FILE, sheet_name="Field_Diagnostics")
duplicate_value_check = pd.read_excel(STANDARDIZED_EXCEL_FILE, sheet_name="Duplicate_Value_Check")
line_snapshot = pd.read_excel(STANDARDIZED_EXCEL_FILE, sheet_name="Line_Snapshot")
common_snapshot = pd.read_excel(STANDARDIZED_EXCEL_FILE, sheet_name="Common_Snapshot")
measurement_long = pd.read_excel(STANDARDIZED_EXCEL_FILE, sheet_name="Measurement_Long")


def safe_display(title, obj, max_rows=10):
    """带标题展示 DataFrame 或对象。"""
    print(f"\n==== {title} ====")
    if isinstance(obj, pd.DataFrame):
        display(obj.head(max_rows))
    else:
        display(obj)


def safe_value_counts(df, col, top_n=10):
    """安全统计某列值分布。列不存在则返回空 DataFrame。"""
    if col not in df.columns:
        print(f"提示：列 `{col}` 不存在，已跳过。")
        return pd.DataFrame(columns=[col, "count"])
    vc = df[col].astype("string").fillna("<NA>").value_counts(dropna=False).head(top_n)
    return vc.rename_axis(col).reset_index(name="count")


def try_parse_time(series):
    """安全解析时间。"""
    # 使用 mixed 格式避免逐元素推断的 warning，并兼容混合时间字符串
    return pd.to_datetime(series, errors="coerce", format="mixed")


def find_columns_by_parameter(df, parameter):
    """
    在 DataFrame 中查找参数列，兼容：
    - 原始 parameter
    - parameter__dup2
    """
    escaped = re.escape(str(parameter))
    pattern = re.compile(rf"^{escaped}(?:__dup\d+)?$")
    return [c for c in df.columns if pattern.match(str(c))]


def get_numeric_summary(df, candidate_columns, category_map=None):
    """对候选列做数值统计。"""
    rows = []
    for col in candidate_columns:
        if col not in df.columns:
            continue
        s = pd.to_numeric(df[col], errors="coerce")
        non_null = int(s.notna().sum())
        if non_null == 0:
            continue
        q = s.quantile([0.25, 0.5, 0.75])
        row = {
            "Parameter": col,
            "Category": (category_map or {}).get(col),
            "non_null_count": non_null,
            "mean": s.mean(),
            "std": s.std(),
            "min": s.min(),
            "p25": q.loc[0.25],
            "median": q.loc[0.5],
            "p75": q.loc[0.75],
            "max": s.max(),
        }
        rows.append(row)
    return pd.DataFrame(rows)

# 1) 标准化数据规模总览
table_summary = pd.DataFrame([
    {"table_name": "Field_Map", "rows": field_map.shape[0], "columns": field_map.shape[1], "purpose": "字段映射与参数分类"},
    {"table_name": "Field_Diagnostics", "rows": field_diagnostics.shape[0], "columns": field_diagnostics.shape[1], "purpose": "字段质量诊断"},
    {"table_name": "Duplicate_Value_Check", "rows": duplicate_value_check.shape[0], "columns": duplicate_value_check.shape[1], "purpose": "重复字段数值检查"},
    {"table_name": "Line_Snapshot", "rows": line_snapshot.shape[0], "columns": line_snapshot.shape[1], "purpose": "每个时间点、每条线一行"},
    {"table_name": "Common_Snapshot", "rows": common_snapshot.shape[0], "columns": common_snapshot.shape[1], "purpose": "每个时间点、设备公共参数一行"},
    {"table_name": "Measurement_Long", "rows": measurement_long.shape[0], "columns": measurement_long.shape[1], "purpose": "适合趋势、异常、AI 洞察的指标长表"},
])
safe_display("标准化数据规模总览", table_summary, max_rows=20)

# 2) 自动识别到的线体
if "Line" in line_snapshot.columns:
    tmp = line_snapshot.copy()
    tmp["_parsed_time"] = try_parse_time(tmp["Time"]) if "Time" in tmp.columns else pd.NaT
    line_counts = tmp.groupby("Line", dropna=False).agg(
        row_count=("Line", "size"),
        start_time=("_parsed_time", "min"),
        end_time=("_parsed_time", "max"),
    ).reset_index().sort_values("row_count", ascending=False)
else:
    print("提示：Line_Snapshot 不含 `Line` 列，无法按线体统计。")
    line_counts = pd.DataFrame(columns=["Line", "row_count", "start_time", "end_time"])
safe_display("自动识别线体统计", line_counts, max_rows=20)

# 3) 字段结构分析
scope_counts = field_map["scope"].value_counts(dropna=False).rename_axis("scope").reset_index(name="count") if "scope" in field_map.columns else pd.DataFrame(columns=["scope", "count"])
category_counts = field_map["Category"].value_counts(dropna=False).rename_axis("Category").reset_index(name="count") if "Category" in field_map.columns else pd.DataFrame(columns=["Category", "count"])

if set(["scope", "line_id", "Category"]).issubset(field_map.columns):
    line_category_count = (
        field_map[field_map["scope"] == "line"]
        .groupby(["line_id", "Category"], dropna=False)
        .size()
        .reset_index(name="field_count")
        .sort_values(["line_id", "field_count"], ascending=[True, False])
    )
else:
    line_category_count = pd.DataFrame(columns=["line_id", "Category", "field_count"])

safe_display("scope 数量统计", scope_counts, max_rows=20)
safe_display("Category 数量统计", category_counts, max_rows=30)
safe_display("线体字段数量统计（line_id + Category）", line_category_count, max_rows=30)

# 4) 字段质量诊断摘要
if field_diagnostics.empty:
    print("未发现字段诊断问题")
    diagnostic_counts = pd.DataFrame(columns=["issue_type", "count"])
    key_diagnostic_issues = pd.DataFrame()
else:
    diagnostic_counts = field_diagnostics["issue_type"].value_counts(dropna=False).rename_axis("issue_type").reset_index(name="count") if "issue_type" in field_diagnostics.columns else pd.DataFrame(columns=["issue_type", "count"])
    keywords = ["Strike OnOff", "Oven(1) TC", "DI(1) Pressure", "Pressure", "TC"]
    joined = field_diagnostics.fillna("").apply(lambda row: " | ".join(map(str, row.values.tolist())), axis=1)
    pattern = "|".join(re.escape(k) for k in keywords)
    key_diagnostic_issues = field_diagnostics[joined.str.contains(pattern, case=False, na=False)]

safe_display("字段诊断 issue_type 统计", diagnostic_counts, max_rows=20)
safe_display("字段诊断重点问题（关键词过滤）", key_diagnostic_issues, max_rows=20)

# 5) 重复字段数值检查
if duplicate_value_check.empty:
    print("未发现重复字段")
    duplicate_assessment_counts = pd.DataFrame(columns=["duplicate_value_assessment", "count"])
    key_duplicate_rows = pd.DataFrame()
else:
    if "duplicate_value_assessment" in duplicate_value_check.columns:
        duplicate_assessment_counts = (
            duplicate_value_check["duplicate_value_assessment"]
            .value_counts(dropna=False)
            .rename_axis("duplicate_value_assessment")
            .reset_index(name="count")
        )
    else:
        duplicate_assessment_counts = pd.DataFrame(columns=["duplicate_value_assessment", "count"])
    keywords_dup = ["Oven(1) TC", "DI(1) Pressure"]
    joined_dup = duplicate_value_check.fillna("").apply(lambda row: " | ".join(map(str, row.values.tolist())), axis=1)
    pattern_dup = "|".join(re.escape(k) for k in keywords_dup)
    key_duplicate_rows = duplicate_value_check[joined_dup.str.contains(pattern_dup, case=False, na=False)]

safe_display("重复字段数值检查评估统计", duplicate_assessment_counts, max_rows=20)
safe_display("重复字段重点展示（关键词过滤）", key_duplicate_rows, max_rows=20)

# 6) 线体运行状态探索
for col in ["Mode", "Product Name", "Lot ID", "Tool ID", "Electrode ID"]:
    safe_display(f"{col} 值分布 Top10", safe_value_counts(line_snapshot, col, top_n=10), max_rows=10)

if set(["Line", "Mode"]).issubset(line_snapshot.columns):
    line_mode_counts = (
        line_snapshot.groupby(["Line", "Mode"], dropna=False)
        .size()
        .reset_index(name="count")
        .sort_values("count", ascending=False)
    )
    safe_display("按 Line + Mode 统计", line_mode_counts, max_rows=20)
else:
    print("提示：缺少 Line 或 Mode 列，无法输出 Line + Mode 分布。")

# 7) 数值参数探索：Line_Snapshot
numeric_categories = {"speed", "current", "voltage", "temperature", "flow", "pressure", "conductivity", "ph", "level"}
if set(["scope", "Parameter", "Category"]).issubset(field_map.columns):
    line_map = field_map[(field_map["scope"] == "line") & (field_map["Category"].isin(numeric_categories))].copy()
else:
    line_map = pd.DataFrame(columns=["Parameter", "Category"])

line_rows = []
for _, r in line_map.drop_duplicates(subset=["Parameter", "Category"]).iterrows():
    param = r["Parameter"]
    cat = r["Category"]
    matched_cols = find_columns_by_parameter(line_snapshot, param)
    for mc in matched_cols:
        s = pd.to_numeric(line_snapshot[mc], errors="coerce")
        nn = int(s.notna().sum())
        if nn == 0:
            continue
        q = s.quantile([0.25, 0.5, 0.75])
        row = {
            "Parameter": mc,
            "Category": cat,
            "non_null_count": nn,
            "mean": s.mean(),
            "std": s.std(),
            "min": s.min(),
            "p25": q.loc[0.25],
            "median": q.loc[0.5],
            "p75": q.loc[0.75],
            "max": s.max(),
        }
        if "Line" in line_snapshot.columns:
            row["line_count_with_data"] = int(line_snapshot.loc[s.notna(), "Line"].nunique())
        else:
            row["line_count_with_data"] = np.nan
        line_rows.append(row)

line_numeric_summary = pd.DataFrame(line_rows).sort_values("std", ascending=False) if line_rows else pd.DataFrame(columns=[
    "Parameter", "Category", "non_null_count", "mean", "std", "min", "p25", "median", "p75", "max", "line_count_with_data"
])
safe_display("Line_Snapshot 数值参数统计", line_numeric_summary, max_rows=30)

# 8) 公共设备参数探索：Common_Snapshot
common_focus_categories = {"temperature", "ph", "pressure", "flow", "conductivity", "level", "current", "voltage"}
if set(["scope", "Parameter", "Category"]).issubset(field_map.columns):
    common_map = field_map[(field_map["scope"] == "common") & (field_map["Category"].isin(common_focus_categories))].copy()
else:
    common_map = pd.DataFrame(columns=["Parameter", "Category"])

common_rows = []
for _, r in common_map.drop_duplicates(subset=["Parameter", "Category"]).iterrows():
    param = r["Parameter"]
    cat = r["Category"]
    matched_cols = find_columns_by_parameter(common_snapshot, param)
    for mc in matched_cols:
        s = pd.to_numeric(common_snapshot[mc], errors="coerce")
        nn = int(s.notna().sum())
        if nn == 0:
            continue
        q = s.quantile([0.25, 0.5, 0.75])
        common_rows.append({
            "Parameter": mc,
            "Category": cat,
            "non_null_count": nn,
            "mean": s.mean(),
            "std": s.std(),
            "min": s.min(),
            "p25": q.loc[0.25],
            "median": q.loc[0.5],
            "p75": q.loc[0.75],
            "max": s.max(),
        })

common_numeric_summary = pd.DataFrame(common_rows).sort_values("std", ascending=False) if common_rows else pd.DataFrame(columns=[
    "Parameter", "Category", "non_null_count", "mean", "std", "min", "p25", "median", "p75", "max"
])
if common_numeric_summary.empty:
    print("提示：Common_Snapshot 中暂无可分析的公共数值参数。")
safe_display("Common_Snapshot 公共参数统计", common_numeric_summary, max_rows=30)

# 9) Measurement_Long 参数波动排名
if measurement_long.empty:
    variation_rank_top20 = pd.DataFrame(columns=["Scope", "Line", "Parameter", "Category", "count", "mean", "std", "min", "max", "range", "cv"])
else:
    ml = measurement_long.copy()
    ml["Value_Num"] = pd.to_numeric(ml.get("Value"), errors="coerce")
    ml_num = ml[ml["Value_Num"].notna()].copy()
    group_cols = [c for c in ["Scope", "Line", "Parameter", "Category"] if c in ml_num.columns]
    if not group_cols:
        variation_rank_top20 = pd.DataFrame(columns=["Scope", "Line", "Parameter", "Category", "count", "mean", "std", "min", "max", "range", "cv"])
    else:
        var_df = ml_num.groupby(group_cols, dropna=False)["Value_Num"].agg(["count", "mean", "std", "min", "max"]).reset_index()
        var_df["range"] = var_df["max"] - var_df["min"]
        var_df["cv"] = np.where(var_df["mean"].abs() < 1e-12, np.nan, var_df["std"] / var_df["mean"].abs())
        sort_col = "std" if "std" in var_df.columns else "range"
        variation_rank_top20 = var_df.sort_values(sort_col, ascending=False).head(20)

safe_display("Measurement_Long 参数波动排名 Top20", variation_rank_top20, max_rows=20)

# 10) 轻量可视化（matplotlib）
# A. Field Category 数量柱状图
if "Category" in field_map.columns and not field_map.empty:
    fig = plt.figure(figsize=(10, 4))
    category_plot = field_map["Category"].astype("string").fillna("<NA>").value_counts()
    category_plot.plot(kind="bar")
    plt.title("字段 Category 数量分布")
    plt.xlabel("Category")
    plt.ylabel("数量")
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    plt.show()
else:
    print("提示：Field_Map 缺少 Category，已跳过类别柱状图。")

# B. Measurement_Long 参数波动 Top10 柱状图
if not variation_rank_top20.empty:
    fig = plt.figure(figsize=(12, 4))
    top10 = variation_rank_top20.head(10).copy()
    for c in ["Scope", "Line", "Parameter"]:
        if c not in top10.columns:
            top10[c] = ""
    top10["label"] = (
        top10["Scope"].astype("string").fillna("") + "|" +
        top10["Line"].astype("string").fillna("") + "|" +
        top10["Parameter"].astype("string").fillna("")
    )
    y_col = "std" if "std" in top10.columns else "range"
    plt.bar(top10["label"], top10[y_col])
    plt.title("Measurement_Long 参数波动 Top10")
    plt.xlabel("Scope|Line|Parameter")
    plt.ylabel(y_col)
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    plt.show()
else:
    print("提示：variation_rank_top20 为空，已跳过波动 Top10 图。")

# C. Running Speed 趋势图（如果存在）
running_speed_cols = find_columns_by_parameter(line_snapshot, "Running Speed")
if running_speed_cols:
    rs_col = running_speed_cols[0]
    plot_df = line_snapshot.copy()
    plot_df["_y"] = pd.to_numeric(plot_df[rs_col], errors="coerce")
    if "Time" in plot_df.columns:
        plot_df["_x"] = try_parse_time(plot_df["Time"])
    else:
        plot_df["_x"] = pd.NaT

    fig = plt.figure(figsize=(12, 4))
    if "Line" in plot_df.columns:
        for ln, grp in plot_df.groupby("Line", dropna=False):
            x = grp["_x"] if grp["_x"].notna().any() else grp.index
            plt.plot(x, grp["_y"], label=str(ln))
        plt.legend(title="Line")
    else:
        x = plot_df["_x"] if plot_df["_x"].notna().any() else plot_df.index
        plt.plot(x, plot_df["_y"])
    plt.title("Running Speed 趋势")
    plt.xlabel("时间（无法解析则使用索引）")
    plt.ylabel("Running Speed")
    plt.tight_layout()
    plt.show()
else:
    print("提示：Line_Snapshot 中不存在 Running Speed，已跳过趋势图。")

# 后续模块衔接变量
eda_summary_tables = {
    "table_summary": table_summary,
    "line_counts": line_counts,
    "scope_counts": scope_counts,
    "category_counts": category_counts,
    "diagnostic_counts": diagnostic_counts,
    "duplicate_assessment_counts": duplicate_assessment_counts,
    "line_numeric_summary": line_numeric_summary,
    "common_numeric_summary": common_numeric_summary,
    "variation_rank_top20": variation_rank_top20,
}

display(Markdown(
    """
**后续异常检测和 AI 洞察建议优先使用：**

- `measurement_long`：做参数级异常检测和波动排名
- `line_snapshot`：做线体对比和生产上下文解释
- `common_snapshot`：做公共设备状态解释
- `field_diagnostics / duplicate_value_check`：解释数据质量问题
"""
))


In [ ]:
# EDA 已在上一单元直接执行并产出 eda_summary_tables，保留该单元避免编号变更。


## 3) 特征选择与清洗层（基于标准化文件，而非原始宽表）
本层**不再直接读取原始设备 Log**，而是读取标准化后的数据文件：`data/processed_equipment_log_standardized.xlsx`。

这样可确保后续分析建立在统一字段口径之上，避免 L1/L2 字段割裂带来的统计偏差。

本层作用：
1. 从标准化数据中识别可分析字段；
2. 过滤明显不适合建模的字段；
3. 处理缺失值、常数列、重复列等数据质量问题；
4. 形成后续 EDA / 异常检测 / 良率影响分析 / AI 洞察可复用特征集。



In [ ]:
def clean_data(df, selected_cols=None):
    # 中文说明：构建 df_clean（分析中间层）
    # 目标：让后续特征工程和异常检测都基于“可解释、可复用、可上线”的数据。
    df = df.copy()

    if selected_cols is None:
        # 中文说明：按制造场景常见重要变量做优先匹配
        priority_patterns = [
            ["speed"], ["ph"], ["pressure"], ["temp"], ["temperature"],
            ["flow"], ["current"], ["voltage"], ["torque"], ["vibration"], ["rpm"]
        ]

        selected_cols = []
        for p in priority_patterns:
            col = _find_column(df, p)
            if col and col not in selected_cols:
                selected_cols.append(col)
            if len(selected_cols) >= 10:
                break

    if not selected_cols:
        raise ValueError("未选中可用于清洗的列。")

    df_clean = df[selected_cols].copy()

    # 中文说明：字段标准化，避免中文/符号/空格导致脚本不稳定
    def normalize_col(c):
        c = c.strip().lower()
        c = re.sub(r"[^a-z0-9]+", "_", c)
        c = re.sub(r"_+", "_", c).strip("_")
        return c

    rename_map = {c: normalize_col(c) for c in df_clean.columns}
    df_clean = df_clean.rename(columns=rename_map)

    # 中文说明：尽量转为数值列，无法转换的记为 NaN
    for c in df_clean.columns:
        df_clean[c] = pd.to_numeric(df_clean[c], errors="coerce")

    # 中文说明：缺失值处理策略
    # 1) ffill/bfill 保留时间连续性
    # 2) 若仍有缺失，用中位数兜底（抗异常值能力比均值更好）
    df_clean = df_clean.ffill().bfill()
    for c in df_clean.columns:
        if df_clean[c].isna().any():
            df_clean[c] = df_clean[c].fillna(df_clean[c].median())

    print(f"df_clean 数据形状： {df_clean.shape}")
    print("字段列表：", list(df_clean.columns))
    return df_clean

In [ ]:
# 基于标准化后的文件进行特征选择与清洗，不再回退原始宽表。
from pathlib import Path

if 'DATA_DIR' not in globals():
    DATA_DIR = Path('data')

PROCESSED_DATA_PATH = DATA_DIR / 'processed_equipment_log_standardized.xlsx'

if not PROCESSED_DATA_PATH.exists():
    raise FileNotFoundError(
        f"未找到标准化后的数据文件：{PROCESSED_DATA_PATH}\n"
        "请先运行前面的数据标准化处理 Cell，生成 processed_equipment_log_standardized.xlsx。"
    )

print(f"使用标准化后的数据文件进行特征选择与清洗：{PROCESSED_DATA_PATH}")
df = pd.read_excel(PROCESSED_DATA_PATH)
print(f"标准化数据维度：{df.shape[0]} 行 × {df.shape[1]} 列")

df_clean = clean_data(df)
display(df_clean.head())


### 后续用途说明（标准化数据链路价值）
标准化数据与清洗后特征集可直接用于：
1. 生产过程 EDA；
2. L1 / L2 工艺参数对比；
3. 异常时段识别；
4. 关键影响因子筛选；
5. 良率波动原因分析；
6. AI 洞察报告生成；
7. 后续接入 MES / AI 分析平台的数据底座验证。

这个 Notebook 的价值不只是完成 Excel 转换，而是建立一条从设备 Log 到 AI 良率洞察的可追溯数据链路。


## 4) 特征工程（特征工程）
我们把原始点位转换成“更接近工艺行为”的特征：

- `speed_diff`：速度变化强度（识别节拍波动）；
- `pH_range`：滚动窗口内 pH 振幅（识别化学稳定性风险）；
- `pressure_variation`：滚动压力波动（识别设备/工艺应力波动）。

**业务意义**：
客户真正关心的是“过程是否稳定”，不是单个点位瞬时值本身。

In [ ]:
# ===== 中文注释：以下代码可直接用于客户演示，强调“可解释、可落地、可复用” =====
def engineer_features(df_clean):
    df_feat = df_clean.copy()

    speed_col = _find_column(df_feat, ["speed"]) or _find_column(df_feat, ["rpm"])
    ph_col = _find_column(df_feat, ["ph"])
    pressure_col = _find_column(df_feat, ["pressure"])

    if speed_col is None:
        raise ValueError("未找到可用于 speed_diff 的速度类字段。")
    if ph_col is None:
        raise ValueError("未找到可用于 pH_range 的 pH 字段。")
    if pressure_col is None:
        raise ValueError("未找到可用于 pressure_variation 的压力字段。")

    df_feat["speed_diff"] = df_feat[speed_col].diff().abs().fillna(0)

    roll_window = 12
    ph_roll_max = df_feat[ph_col].rolling(roll_window, min_periods=1).max()
    ph_roll_min = df_feat[ph_col].rolling(roll_window, min_periods=1).min()
    df_feat["pH_range"] = ph_roll_max - ph_roll_min

    df_feat["pressure_variation"] = (
        df_feat[pressure_col].rolling(roll_window, min_periods=2).std().fillna(0)
    )

    return df_feat

In [ ]:
df_feat = engineer_features(df_clean)
display(df_feat[["speed_diff", "pH_range", "pressure_variation"]].head(15))

## 5) 异常检测（异常检测）
在没有良率标签时，我们先识别“异常工况时段”。

方法：
- 对每个数值特征做 Z分数；
- 聚合为 `anomaly_score`；
- 用分位数阈值得到 `anomaly_flag`。

**对客户的解释话术**：
这不是替代 SPC 规则，而是提供一层“无监督 AI 雷达”，帮助快速定位需要优先排查的时间段。

In [ ]:
# ===== 中文注释：以下代码可直接用于客户演示，强调“可解释、可落地、可复用” =====
def detect_anomaly(df_feat, threshold_quantile=0.97):
    df_out = df_feat.copy()
    num_cols = df_out.select_dtypes(include=[np.number]).columns.tolist()

    z_dict = {}
    for c in num_cols:
        mean = df_out[c].mean()
        std = df_out[c].std(ddof=0)
        if std == 0 or np.isnan(std):
            z_dict[c] = np.zeros(len(df_out))
        else:
            z_dict[c] = (df_out[c] - mean) / std

    z_df = pd.DataFrame(z_dict, index=df_out.index)
    df_out["anomaly_score"] = z_df.abs().mean(axis=1)

    cutoff = df_out["anomaly_score"].quantile(threshold_quantile)
    df_out["anomaly_flag"] = (df_out["anomaly_score"] >= cutoff).astype(int)

    print(f"异常阈值（分位数 q={threshold_quantile:.2f}）：{cutoff:.3f}")
    print(df_out["anomaly_flag"].value_counts().rename({0: "正常", 1: "AB正常"}))
    return df_out

In [ ]:
df_anom = detect_anomaly(df_feat, threshold_quantile=0.97)

def plot_anomalies(df, y_col="anomaly_score"):
    # 中文说明：绘制异常分数，并高亮异常段，便于快速定位风险时段
    plt.figure(figsize=(14, 4))
    plt.plot(df.index, df[y_col], label=cn_label(y_col), linewidth=1.2)

    abnormal_idx = df.index[df["anomaly_flag"] == 1]
    plt.scatter(abnormal_idx, df.loc[abnormal_idx, y_col], color="red", s=18, label="异常点", alpha=0.8)
    plt.title("异常检测结果")
    plt.xlabel("记录序号")
    plt.ylabel(cn_label(y_col))
    plt.legend()
    plt.tight_layout()
    plt.show()

plot_anomalies(df_anom)


## 6) 分段对比（核心洞察）
将数据分为两段：
- 正常段
- 异常段

然后对比均值并按差异排序，得到“最值得关注的变量”。

**为什么这一步接近良率分析？**
虽然没有直接良率标签，但异常段常常对应过程偏离。比较两段差异，可以近似识别“潜在影响良率”的工艺因子。


In [ ]:
# ===== 中文注释：以下代码可直接用于客户演示，强调“可解释、可落地、可复用” =====
def compare_segments(df_anom, top_n=10):
    num_cols = df_anom.select_dtypes(include=[np.number]).columns.tolist()
    num_cols = [c for c in num_cols if c not in ["anomaly_flag"]]

    grp = df_anom.groupby("anomaly_flag")[num_cols].mean().T
    grp.columns = ["normal_mean", "abnormal_mean"]

    # 中文说明：增加中文展示字段，便于客户会上直接阅读

    grp["abs_diff"] = (grp["abnormal_mean"] - grp["normal_mean"]).abs()
    grp["pct_diff_vs_normal"] = np.where(
        grp["normal_mean"].abs() > 1e-12,
        (grp["abnormal_mean"] - grp["normal_mean"]) / grp["normal_mean"] * 100,
        np.nan,
    )

    ranked = grp.sort_values("abs_diff", ascending=False)
    ranked_display = ranked.rename(index=lambda x: cn_label(x), columns=lambda x: cn_label(x))
    return ranked_display.head(top_n), ranked_display

In [ ]:
top_diff, full_diff = compare_segments(df_anom, top_n=10)
print("正常段与异常段差异最大的变量（Top10）：")
display(top_diff)


## 7) AI 文本洞察生成
我们把统计差异自动转成现场可读的结论：
- 哪些变量变化最大；
- 变化方向是什么；
- 建议先做哪些工程检查。

可选：调用本地 Ollama 生成更自然的中文客户报告文案。


In [ ]:
# ===== 中文注释：以下代码可直接用于客户演示，强调“可解释、可落地、可复用” =====
def generate_insight(top_diff, max_items=5):
    # 中文说明：将差异结果转成可直接口播的洞察文本
    lines = []
    lines.append("AI洞察总结（正常 vs 异常）")
    lines.append("-" * 50)

    focus = top_diff.head(max_items).reset_index().rename(columns={"index": "feature"})
    for i, row in focus.iterrows():
        direction = "更高" if row["异常段均值"] > row["正常段均值"] else "更低"
        lines.append(
            f"{i+1}. {row['feature']}：异常段相对正常段{direction} "
            f"（正常={row['正常段均值']:.3f}，异常={row['异常段均值']:.3f}，"
            f"变化={row['相对正常段变化(%)']:.1f}%）。"
        )

    lines.append("\n建议优先检查：")
    lines.append("- 核对高差异变量的控制界限与设定值漂移。")
    lines.append("- 将异常时段与维保记录、换批/换线事件进行交叉比对。")
    lines.append("- 将这些变量作为根因分析与DOE实验设计的优先因子。")
    return "\n".join(lines)


def generate_insight_with_ollama(prompt, model="gemma4:e4b", url="http://localhost:11434/api/generate"):
    """
    可选能力：调用本地 Ollama 模型，生成更自然的客户汇报文案。
    需要先执行 `ollama serve` 并提前拉取模型。
    """
    import requests

    payload = {"model": model, "prompt": prompt, "stream": False}
    resp = requests.post(url, json=payload, timeout=150)
    resp.raise_for_status()
    return resp.json().get("response", "")


In [ ]:

# ===== 基础规则洞察 =====
insight_text = generate_insight(top_diff, max_items=5)

print("\n=== 规则生成洞察 ===")
print(insight_text)


# ===== AI增强洞察（自动fallback） =====
def safe_llm_insight(insight_text):
    # 中文说明：如本地LLM不可用，自动回退到规则洞察，保证演示不断档
    try:
        llm_prompt = f"""
你是一名制造工艺专家。

请将下面的分析结果整理为简洁、专业、适合客户汇报的中文洞察。
重点覆盖：
- 关键异常因子
- 可能的工艺影响
- 建议优先关注方向

分析内容：
{insight_text}
"""
        ai_text = generate_insight_with_ollama(
            llm_prompt,
            model="gemma4:e4b"  # 或你实际模型
        )
        return ai_text.strip()

    except Exception as e:
        return f"[LLM不可用] 已回退为规则洞察。\n错误信息：{str(e)}"


# ===== 输出AI洞察 =====
print("\n=== AI洞察（LLM增强）===")
ai_insight = safe_llm_insight(insight_text)
print(ai_insight)

print("\n" + "="*50)
print("最终洞察对比")
print("="*50)

print("\n[1] 规则生成洞察：\n")
print(insight_text)

print("\n[2] AI增强洞察：\n")
print(ai_insight)


## 8) 最终结论（给管理层）
- 这套方法可补足 MES 分析中的一部分“早期诊断”能力；
- 不需要良率标签也能先做过程风险定位；
- 适合快速 PoC，再逐步接入更多系统（MES/QMS/维保事件）。

**落地收益**：
上线快、投入轻、见效早，适合中国制造客户先小步试点再规模复制。


## 9) 演示讲解要点（客户演示讲稿）
1. **先解决 80% 的问题，再谈大平台**：不用等 MES 全部打通，先从设备日志直接产出洞察。  
2. **没有良率标签也能做**：通过异常段对比，先找出高风险变量，提升排查效率。  
3. **结果可解释，工程师听得懂**：趋势图 + 差异表 + 中文洞察建议。  
4. **技术栈轻量，信息化 阻力小**：pandas/numpy/matplotlib 即可启动。  
5. **可平滑升级**：后续接入良率、批次、维保记录后，模型效果会持续增强。

---

### 总结
> 我们这套方法不是替代 MES，而是让您在 MES 完整建设前，就先拿到可行动的工艺洞察。  
> 先快速看到异常、定位重点变量、缩小排查范围，再决定下一步系统化投资，风险更低、回报更快。

## 本演示价值总结
- **没有MES也能发现问题**：直接利用设备日志做数据清洗、异常识别与分段对比，快速定位疑似风险工况。
- **AI辅助工程师提效**：先用规则洞察确保稳定输出，再用LLM生成客户可读总结，减少手工整理时间。
- **后续可扩展方向**：逐步接入良率、批次、维保与换线事件；从“发现异常”升级到“预测风险与闭环优化”。


本 Notebook 已将原始设备宽表日志标准化为线体快照、公共设备快照和指标长表。该结构不再依赖固定列号，能够兼容一机两线、一机三线或更多线体，并能自动识别重复字段、线体字段不一致和公共设备参数，为后续良率分析、异常检测和 AI 洞察提供可靠的数据基础。
